# Manual Testing: QueryOrchestrator → TavilyRetriever Pipeline

This notebook provides hands-on testing of the integrated QueryOrchestrator and TavilyRetriever agents using real API keys.

## Pipeline Flow
1. **QueryOrchestrator** - Parses user query → structured search query + intent
2. **TavilyRetriever** - Executes Tavily two-step process (search → extract)
3. **State Management** - Track agent execution and results

## Requirements
- Real OPENAI_API_KEY and TAVILY_API_KEY in .env file
- Backend dependencies installed

## Setup and Imports

In [ ]:
import os
import sys
import asyncio
import json
from datetime import datetime
from pprint import pprint

# Add backend to path
sys.path.append('..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

print("✅ Environment loaded")
print(f"OPENAI_API_KEY: {'✅ Set' if os.getenv('OPENAI_API_KEY') else '❌ Missing'}")
print(f"TAVILY_API_KEY: {'✅ Set' if os.getenv('TAVILY_API_KEY') else '❌ Missing'}")

In [ ]:
# Import our agents and state management
from app.agents.query_orchestrator_agent import QueryOrchestratorAgent
from app.agents.tavily_retriever_agent import TavilyRetrieverAgent
from app.agents.state import create_initial_state, get_state_summary
from app.config import settings

print("✅ Agents imported successfully")
print(f"Environment: {settings.ENVIRONMENT}")
print(f"OpenAI Model: {settings.OPENAI_MODEL}")

## Initialize Agents

In [ ]:
# Create agent instances
query_agent = QueryOrchestratorAgent()
tavily_agent = TavilyRetrieverAgent()

print("✅ Agents initialized:")
print(f"- {query_agent.name} (OpenAI: {query_agent.llm.model_name})")
print(f"- {tavily_agent.name} (Client: {type(tavily_agent.tavily_client).__name__})")

## Test Configuration

In [ ]:
# Test Tavily configuration
config_test = await tavily_agent.test_configuration()
print("🔧 Tavily Configuration:")
pprint(config_test)

## Manual Test Functions

In [ ]:
async def test_query_pipeline(raw_query: str, run_id: str = None):
    """
    Complete pipeline test: QueryOrchestrator → TavilyRetriever
    """
    if not run_id:
        run_id = f"manual_test_{datetime.now().strftime('%H%M%S')}"
    
    print(f"\n🚀 Testing Query: '{raw_query}'")
    print("=" * 60)
    
    # Create initial state
    state = create_initial_state(raw_query=raw_query, run_id=run_id)
    print(f"📋 Initial state created (run_id: {run_id})")
    
    try:
        # Step 1: QueryOrchestrator
        print("\n1️⃣ QueryOrchestrator Processing...")
        start_time = datetime.now()
        state = await query_agent.process(state)
        query_time = (datetime.now() - start_time).total_seconds()
        
        search_query = state.get("search_query")
        if search_query:
            print(f"✅ Query parsed successfully ({query_time:.2f}s)")
            print(f"   Intent: {search_query.intent}")
            print(f"   Category: {search_query.category}")
            print(f"   Normalized: '{search_query.normalized_query}'")
            if search_query.budget_max:
                print(f"   Budget: ${search_query.budget_max}")
            if search_query.constraints:
                print(f"   Constraints: {search_query.constraints}")
                
            # Show Tavily parameters
            tavily_params = state.get("tavily_search_params", {})
            print(f"   Tavily params: {tavily_params}")
        else:
            print("❌ Query parsing failed")
            return state
        
        # Step 2: TavilyRetriever
        print("\n2️⃣ TavilyRetriever Processing...")
        start_time = datetime.now()
        state = await tavily_agent.process(state)
        tavily_time = (datetime.now() - start_time).total_seconds()
        
        # Analyze results
        search_results = state.get("raw_search_results", [])
        extracted_content = state.get("extracted_content", [])
        coverage_score = state.get("coverage_score", 0.0)
        
        print(f"✅ Tavily processing completed ({tavily_time:.2f}s)")
        print(f"   Search results: {len(search_results)}")
        print(f"   Extracted content: {len(extracted_content)}")
        print(f"   Coverage score: {coverage_score:.2f}")
        
        # Show agent execution summary
        summary = get_state_summary(state)
        print(f"\n📊 Execution Summary:")
        print(f"   Total time: {query_time + tavily_time:.2f}s")
        print(f"   Agents completed: {summary['progress']['agents_completed']}")
        print(f"   Total cost: ${summary['progress']['total_cost_usd']:.4f}")
        
        return state
        
    except Exception as e:
        print(f"❌ Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        return state

def show_search_results(state: dict, max_results: int = 3):
    """
    Display search results in a readable format
    """
    search_results = state.get("raw_search_results", [])
    
    if not search_results:
        print("No search results found")
        return
    
    # Ensure search_results is a list
    if isinstance(search_results, dict):
        search_results = search_results.get("results", [])
    
    if not isinstance(search_results, list):
        print(f"Error: search_results is not a list, got {type(search_results)}")
        return
    
    print(f"\n🔍 Search Results (showing {min(max_results, len(search_results))} of {len(search_results)}):")
    print("=" * 60)
    
    for i, result in enumerate(search_results[:max_results]):
        print(f"\n{i+1}. {result.get('title', 'No title')}")
        print(f"   URL: {result.get('url', 'No URL')}")
        content = result.get('content', 'No content')
        if len(content) > 150:
            content = content[:150] + "..."
        print(f"   Content: {content}")
        if 'score' in result:
            print(f"   Score: {result['score']:.3f}")

def show_extracted_content(state: dict, max_content: int = 2):
    """
    Display extracted content in a readable format
    """
    extracted_content = state.get("extracted_content", [])
    
    if not extracted_content:
        print("No extracted content found")
        return
    
    # Ensure extracted_content is a list
    if not isinstance(extracted_content, list):
        print(f"Error: extracted_content is not a list, got {type(extracted_content)}")
        return
    
    print(f"\n📄 Extracted Content (showing {min(max_content, len(extracted_content))} of {len(extracted_content)}):")
    print("=" * 60)
    
    for i, content in enumerate(extracted_content[:max_content]):
        print(f"\n{i+1}. URL: {content.get('url', 'No URL')}")
        extracted_text = content.get('content', 'No content')
        if len(extracted_text) > 300:
            extracted_text = extracted_text[:300] + "..."
        print(f"   Content: {extracted_text}")
        if 'quality_score' in content:
            print(f"   Quality: {content['quality_score']:.3f}")

## Interactive Testing

Now you can test different queries manually. Try the examples below or create your own!

### Test 1: Gaming Laptop Search

In [ ]:
# Test gaming laptop search
result_state = await test_query_pipeline("best gaming laptop under $2000")

In [ ]:
# Show detailed results
show_search_results(result_state, max_results=5)
show_extracted_content(result_state, max_content=3)

### Test 2: Product Comparison

In [ ]:
# Test product comparison
result_state = await test_query_pipeline("iPhone 15 vs Samsung Galaxy S24 camera quality")

In [ ]:
# Show detailed results
show_search_results(result_state, max_results=4)
show_extracted_content(result_state, max_content=2)

### Test 3: Review Search

In [ ]:
# Test review search
result_state = await test_query_pipeline("Sony WH-1000XM5 headphones review")

In [ ]:
# Show detailed results
show_search_results(result_state, max_results=3)
show_extracted_content(result_state, max_content=2)

### Custom Query Testing

Use this cell to test your own queries:

In [ ]:
# Your custom query here
custom_query = "mechanical keyboard for programming Cherry MX switches"
custom_result = await test_query_pipeline(custom_query)

In [ ]:
# Show your custom results
show_search_results(custom_result, max_results=3)
show_extracted_content(custom_result, max_content=2)

## Deep Dive: State Analysis

Examine the complete state structure and agent execution details:

In [ ]:
# Analyze the complete state from your last test
print("🔍 Complete State Analysis:")
print("=" * 60)

# Show state keys
print(f"State keys: {list(custom_result.keys())}")

# Show search query details
search_query = custom_result.get("search_query")
if search_query:
    print(f"\n📝 Parsed Query:")
    pprint(search_query.model_dump())

# Show agent execution steps
agent_steps = custom_result.get("agent_steps", [])
print(f"\n🏃 Agent Execution Steps ({len(agent_steps)}):")
for step in agent_steps:
    print(f"  {step.agent_name}: {step.status} ({step.execution_time_ms}ms, ${step.cost_usd:.4f})")

## Performance Testing

Test multiple queries to understand performance patterns:

In [ ]:
# Performance test with multiple queries
test_queries = [
    "wireless earbuds under $100",
    "RTX 4070 graphics card review",
    "MacBook Air vs ThinkPad comparison",
    "best monitor for productivity 4K"
]

performance_results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}/{len(test_queries)}: Testing '{query}'")
    start_time = datetime.now()
    
    result = await test_query_pipeline(query, f"perf_test_{i}")
    
    total_time = (datetime.now() - start_time).total_seconds()
    summary = get_state_summary(result)
    
    performance_results.append({
        "query": query,
        "time_seconds": total_time,
        "cost_usd": summary['progress']['total_cost_usd'],
        "search_results": len(result.get("raw_search_results", [])),
        "extracted_content": len(result.get("extracted_content", []))
    })

print("\n📊 Performance Summary:")
print("=" * 60)
for result in performance_results:
    print(f"Query: {result['query'][:40]}...")
    print(f"  Time: {result['time_seconds']:.1f}s, Cost: ${result['cost_usd']:.4f}")
    print(f"  Results: {result['search_results']} search, {result['extracted_content']} extracted\n")

## Experimentation Area

Use this space to experiment with different aspects of the pipeline:

In [ ]:
# Experiment with edge cases
edge_cases = [
    "laptop",  # Very simple query
    "best affordable gaming laptop with RTX 4060 under $1200 for programming and gaming",  # Complex query
    "xyz123 unknown product",  # Non-existent product
    ""  # Empty query
]

print("🧪 Testing Edge Cases:")
for query in edge_cases:
    if not query:
        query = "[empty string]"
    print(f"\nTesting: '{query}'")
    try:
        result = await test_query_pipeline(query if query != "[empty string]" else "")
        print("✅ Handled successfully")
    except Exception as e:
        print(f"❌ Error: {e}")

## Next Steps

After testing this pipeline, the next agent to implement is:
- **CredibilityFilterAgent** - Filter and score results by domain credibility and recency

The complete pipeline will be:
`QueryOrchestrator → RetrievalSplitter → TavilyRetriever → CredibilityFilter → SpecExtractor → ResultsRanker`